In [0]:
%run ./01-config

In [0]:
class Silver():
    def __init__(self, env, db_name):
        Conf = Config()
        self.landing_zone = Conf.base_dir_data + "/raw"
        self.checkpoint_base = Conf.base_dir_data + "/checkpoints"
        self.catalog = env
        self.db_name = db_name
        spark.sql(f"USE {self.catalog}.{self.db_name}")

    def consume_bronze_crm_cust_info(self):
        from pyspark.sql import functions as F
        #schema = "cst_id INT, cst_key string, cst_firstname string, cst_lastname string, cst_marital_status string, cst_gndr string, cst_create_date DATE"

        df = (spark.read
                .format("delta")
                .table(f"{self.catalog}.{self.db_name}.crm_cust_info}")
                .withColumn("dlh_loaded_date", F.current_timestamp())
                .withColumn("cst_key", F.trim(F.col("cst_key")))
                .withColumn("cst_firstname", F.trim(F.col("cst_firstname")))
                .withColumn("cst_lastname", F.trim(F.col("cst_lastname")))
                .withColumn("cst_marital_status", 
                            F.when(F.upper(F.trim(F.col("cst_marital_status"))) == 'M', "Married")
                             .when(F.upper(F.trim(F.col("cst_marital_status"))) == 'S', "Single")
                             .otherwise("n/a")
                )
                .withColumn("cst_gndr", 
                            F.when((F.upper(F.trim(F.col("cst_gndr"))) == 'M') | (F.upper(F.trim(F.col("cst_gndr"))) == 'MALE'), "Male")
                             .when((F.upper(F.trim(F.col("cst_gndr"))) == 'F') | (F.upper(F.trim(F.col("cst_gndr"))) == 'FEMALE'), "Female")
                             .otherwise("n/a")
                )
                .dropDuplicates(F.col("cst_id"))
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.crm_cust_info"))
        
    def consume_bronze_crm_prd_info(self):
        from pyspark.sql import functions as F
        #schema = "prd_id INT, prd_key string, prd_name string, prd_cost INT, prd_line string, prd_start_dt DATE, prd_end_dt DATE"

        df = (spark.read
                .format("delta")
                .table(f"{self.catalog}.{self.db_name}.crm_prd_info")
                .withColumn("dlh_loaded_date", F.current_timestamp())
                .withColumn("cat_id", F.replace(F.substring(F.col("prd_key"), 1, 5), "-", "_"))
                .withColumn("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))
                .withColumn("prd_name", F.trim(F.col("prd_name")))
                .withColumn("prd_cost", 
                            F.when(F.col("prd_cost").isNull(), 0)
                            .otherwise(f.col("prd_cost"))
                            )
                .withColumn("prd_line",
                            F.when(F.upper(F.trim(F.col("prd_line"))) == "M", "Mountain")
                             .when(F.upper(F.trim(F.col("prd_line"))) == "R", "Road")
                             .when(F.upper(F.trim(F.col("prd_line"))) == "S", "Other Sales")
                             .when(F.upper(F.trim(F.col("prd_line"))) == "T", "Touring")
                             .otherwise('n/a')
                            )
                .withColumn("prd_end_dt",
                            F.date_sub(F.lead(F.col("prd_start_dt")).over(Window.partitionBy(F.col("prd_key")).orderBy(F.col("prd_start_dt"))), 1)
                            )        
                )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.crm_prd_info"))
        
    def consume_bronze_crm_sales_details(self):
        from pyspark.sql import functions as F
        #schema = "sls_ord_num string, sls_prd_key	string, sls_cust_id	INT, sls_order_dt INT, sls_ship_dt	INT, sls_due_dt	INT, sls_sales INT, sls_quantity INT, sls_price INT"

        df = (spark.read
                .format("delta")
                .table(f"{self.catalog}.{self.db_name}.crm_sales_details")
                .withColumn("dlh_loaded_date", F.current_timestamp())
                .withColumn("sls_order_dt",
                            F.when((F.col("sls_order_dt") <= 0) | (F.length(F.col("sls_order_dt")) != 8) , None)
                            .otherwise(F.to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd"))
                            )
                .withColumn("sls_ship_dt",
                            F.when((F.col("sls_ship_dt") <= 0) | (F.length(F.col("sls_ship_dt")) != 8), None)
                            .otherwise(F.to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd"))
                            )
                .withColumn("sls_due_dt",
                            F.when((F.col("sls_due_dt") <= 0) | (F.length(F.col("sls_due_dt")) != 8), None)
                            .otherwise(F.to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd"))
                            )
                .withColumn("sls_sales",
                            F.when((F.col("sls_sales") < 0)|(F.col("sls_sales").isNull()), F.col("sls_price") * F.col("sls_quantity"))
                            .otherwise(F.col("sls_sales"))
                            )
                .withColumn("sls_price",
                            F.when((F.col("sls_price") < 0) | (F.col("sls_price").isNull()), F.when(F.col("sls_quantity") > 0, F.col("sls_sales") / F.col("sls_quantity")).otherwise(F.col("sls_quantity")))
                            .otherwise(F.col("sls_price"))
                            )
                            
            )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.crm_sales_details"))
        
    def consume_bronze_erp_cust_az12(self):
        from pyspark.sql import functions as F
        #schema = "cid string, bdate DATE, gen string"

        df = (spark.read
                .format("delta")
                .table(f"{self.catalog}.{self.db_name}.erp_cust_az12")
                .withColumn("dlh_loaded_date", F.current_timestamp())
                .withColumn("cid",
                            F.when(F.col("cid").like('NAS%'), F.replace(F.col("cid"), F.lit("NAS"), F.lit("")))
                            .otherwise(F.col("cid"))
                            )
                .withColumn("bdate",
                            F.when(F.col("dbate") > F.current_date(), None)
                            .otherwise(F.col("dbate"))
                        )
                .withColumn("gen", 
                            F.when((F.upper(F.trim(F.col("gen"))) == 'M') | (F.upper(F.trim(F.col("gen"))) == 'MALE'), "Male")
                             .when((F.upper(F.trim(F.col("gen"))) == 'F') | (F.upper(F.trim(F.col("gen"))) == 'FEMALE'), "Female")
                             .otherwise("n/a")
                )
        )
                

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.erp_cust_az12"))
        
    def consume_bronze_erp_loc_a101(self):
        from pyspark.sql import functions as F
        #schema = "cid string, cntry string"

        df = (spark.read
                .format("delta")
                .table("{self.catalog}.{self.db_name}.erp_loc_a101")
                .withColumn("dlh_loaded_date", F.current_timestamp())
                .withColumn("cntry",
                            F.when(F.upper(F.trim(F.col("cntry"))) == "USA", "United States")
                             .when(F.upper(F.trim(F.col("cntry"))) == "US", "United States")
                             .when(F.upper(F.trim(F.col("cntry"))) == "DE", "Germany")
                             .when(F.upper((F.trim(F.col("cntry"))) == "") |(F.col("cntry").isNull()) , "n/a")
                             .otherwise(F.trim(F.col("cntry")))
                            )
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.erp_loc_a101"))
        
    def consume_bronze_erp_px_cat_glv2(self):
        from pyspark.sql import functions as F
        #schema = "id string, cat string, subcat string, maintenance string"

        df = (spark.read
                .format("delta")
                .table("{self.catalog}.{self.db_name}.erp_px_cat_glv2")
                .withColumn("dlh_loaded_date", F.current_timestamp())
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.erp_px_cat_glv2"))
        
    def consume(self):
        import time
        start = int(time.time())
        print(f"\nStarting silver layer consumption...")
        self.consume_bronze_crm_cust_info()
        self.consume_bronze_crm_prd_info()
        self.consume_bronze_crm_sales_details()
        self.consume_bronze_erp_cust_az12()
        self.consume_bronze_erp_loc_a101()
        self.consume_bronze_erp_px_cat_glv2()
        end = int(time.time())
        print(f"\nCompleted silver layer consumption {end - start} seconds")

    def assert_count(self, table_name, expected_count):
        print(f"\nValidating record counts in {table_name}...", end = '')
        actual_count = spark.read.table(f"{self.catalog}.{self.db_name}.{table_name}").count()
        assert actual_count == expected_count, f"Expected {expected_count:, } records, found {actual_count:,} in {table_name} where {filter}"
        print(f"Found {actual_count:,} / {expected_count:,} records where {filter}: Success")

    def validate(self):
        import time
        start = int(time.time())
        print(f"\nValidating silver layer records...")
        checks = {
        "crm_cust_info",
        "crm_prd_info",
        "crm_sales_details",
        "erp_cust_az12",
        "erp_loc_a101",
        "erp_px_cat_glv2"
        }

        for table in checks:
            src_count = (spark.read.format("delta")
                            .table(f"{table}")
                            .count())
            self.assert_count(table, src_count)
        end = int(time.time())
        print(f"\nSilver layer records validation completed in {end - start} seconds")
